# Fake News Detection — Training Notebook

KZ-ML-02 — TF-IDF + Passive Aggressive Classifier (compared against Multinomial Naive Bayes and Logistic Regression).

Pipeline: load `Fake.csv`/`True.csv` -> remove Reuters dateline leakage -> clean text -> TF-IDF -> train/compare 3 models -> save the best model + vectoriser to `../model/`.

Run this notebook with the `fake-news/` project root as the working directory's parent (it adds `..` to `sys.path` below so the `utils` package can be imported).

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils.data_prep import prepare_dataset, load_raw, LABEL_NAMES
from utils.clean_text import remove_reuters_prefix, clean_text

sns.set_theme(style="whitegrid")

## 1. Load and inspect the raw data

In [ ]:
raw_df = load_raw()
print(raw_df.shape)
raw_df["label"].value_counts().rename(index=LABEL_NAMES)

In [ ]:
raw_df.head()

## 2. The Reuters data-leakage problem

Almost every REAL article starts with a wire-service dateline such as `WASHINGTON (Reuters) - `. If this stays in the text, TF-IDF hands the model a single near-perfect feature (the word "Reuters") instead of making it learn actual writing-style differences — accuracy balloons to ~99%+ for the wrong reason. Compare before/after below.

In [ ]:
sample_real = raw_df[raw_df["label"] == 1]["text"].iloc[0]
print("BEFORE:\n", sample_real[:200])
print("\nAFTER:\n", remove_reuters_prefix(sample_real)[:200])

## 3. Full data-prep pipeline (merge, shuffle, de-leak, clean)

In [ ]:
df = prepare_dataset()
df[["content", "clean_text", "label"]].head()

## 4. Word clouds — most frequent words per class

Useful for the report: shows qualitatively what distinguishes fake vs. real writing style after cleaning.

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, name) in zip(axes, LABEL_NAMES.items()):
    text = " ".join(df.loc[df["label"] == label, "clean_text"])
    wc = WordCloud(width=800, height=400, background_color="white").generate(text)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 5. TF-IDF feature extraction + train/test split

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

vectorizer = TfidfVectorizer(max_df=0.7)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Train shape:", X_train_tfidf.shape, "Test shape:", X_test_tfidf.shape)

## 6. Train and compare 3 models: Passive Aggressive, Naive Bayes, Logistic Regression

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")

from sklearn.linear_model import PassiveAggressiveClassifier, LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

models = {
    "Passive Aggressive": PassiveAggressiveClassifier(max_iter=1000, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
}

results = {}
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)
    results[name] = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
    }
    models[name] = model

pd.DataFrame(results).T

## 7. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test_tfidf)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["FAKE", "REAL"]).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 8. Model comparison bar chart

In [ ]:
results_df = pd.DataFrame(results).T
results_df.plot(kind="bar", figsize=(8, 5), ylim=(0, 1.05))
plt.title("Model Comparison")
plt.ylabel("Score")
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 9. Pick the best model (by F1) and save model + vectoriser

In [ ]:
import joblib
from pathlib import Path

best_name = max(results, key=lambda n: results[n]["f1"])
best_model = models[best_name]
print("Best model:", best_name)
print(classification_report(y_test, best_model.predict(X_test_tfidf), target_names=["FAKE", "REAL"]))

model_dir = Path("../model")
model_dir.mkdir(exist_ok=True)
joblib.dump(best_model, model_dir / "model.pkl")
joblib.dump(vectorizer, model_dir / "tfidf.pkl")
(model_dir / "model_name.txt").write_text(best_name, encoding="utf-8")
print("Saved to", model_dir.resolve())

## 10. Quick manual sanity check

In [ ]:
from utils.clean_text import preprocess
import numpy as np

def predict(text):
    cleaned = preprocess(text)
    vec = vectorizer.transform([cleaned])
    pred = best_model.predict(vec)[0]
    if hasattr(best_model, "decision_function"):
        score = best_model.decision_function(vec)[0]
        confidence = 1 / (1 + np.exp(-abs(score)))
    else:
        confidence = best_model.predict_proba(vec)[0].max()
    label = "REAL" if pred == 1 else "FAKE"
    return label, round(confidence * 100, 2)

predict("Scientists refuse to tell you this one shocking secret, share before it gets deleted!")